# MCP New Cache and Search Tools

This notebook is a runnable smoke test for the newer Perseus MCP search and metadata tools. It uses an in-process FastMCP client, so you do not need to start a separate MCP server process.

It demonstrates:

- cache inspection and refresh;
- paged JSON references and reference counts;
- paginated and server-scoped Scaife search;
- lemma and operator-preserving search;
- reader search within one edition;
- passage highlights;
- Scaife metadata, passage JSON, and passage plaintext tools.

> Requirements: run from the repository root or keep the path setup cell unchanged, install project dependencies, and have internet access to Perseus/Scaife upstream services.


## Setup

The setup cell imports the local `server.py`, reloads it so local edits are visible in an existing kernel, and defines small helpers for calling MCP tools and summarizing search responses.


In [ ]:
from pathlib import Path
import importlib
import json
import sys

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "server.py").exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

from fastmcp import Client
import server

server = importlib.reload(server)
mcp = server.mcp


def tool_text(result):
    return "\n".join(
        block.text for block in result.content if getattr(block, "text", None) is not None
    )


async def call_json(client, tool_name, arguments=None):
    result = await client.call_tool(tool_name, arguments or {})
    return json.loads(tool_text(result))


async def call_text(client, tool_name, arguments=None):
    result = await client.call_tool(tool_name, arguments or {})
    return tool_text(result)


def summarize_search(data):
    results = data.get("results", [])
    first = results[0] if results else {}
    passage = first.get("passage", {})
    text = passage.get("text", {})
    snippet = " ".join(first.get("content", [])) if first else None
    return {
        "total_count": data.get("total_count"),
        "page": data.get("page", {}).get("number"),
        "num_pages": data.get("page", {}).get("num_pages"),
        "first_urn": passage.get("urn"),
        "first_text_label": text.get("label"),
        "first_snippet": snippet,
        "author_scope": data.get("author_scope"),
    }


## Confirm the New Tools Are Registered

This cell checks that the expected newer tool names are present in the MCP tool list. If one is missing, the notebook stops early with an assertion error.


In [ ]:
expected_new_tools = {
    "get_cache_status",
    "refresh_metadata_cache",
    "clear_metadata_cache",
    "get_valid_references_json",
    "count_valid_references",
    "search_within_text",
    "get_passage_highlights",
    "get_scaife_library_metadata",
    "get_scaife_passage_json",
    "get_scaife_passage_text",
}

async with Client(mcp) as client:
    tools = await client.list_tools()

tool_names = {tool.name for tool in tools}
missing = expected_new_tools - tool_names
assert not missing, f"Missing expected tools: {sorted(missing)}"

print(f"Registered tools: {len(tool_names)}")
print("New tools present:")
for name in sorted(expected_new_tools):
    print("-", name)


## Cache Tools

`get_cache_status` reports the in-memory and disk cache state. `refresh_metadata_cache` fetches current CTS capabilities and writes them to the cache. `clear_metadata_cache` is included as an optional guarded cell below because clearing cache files is sometimes useful, but usually not something you want a demo notebook to do automatically.


In [ ]:
async with Client(mcp) as client:
    before = await call_json(client, "get_cache_status")
    refreshed = await call_json(client, "refresh_metadata_cache")
    after = await call_json(client, "get_cache_status")

assert "cache_dir" in after
assert after["disk_files"] >= before["disk_files"]

print("Before:")
print(json.dumps(before, ensure_ascii=False, indent=2))
print("\nRefresh result:")
print(json.dumps(refreshed, ensure_ascii=False, indent=2)[:1000])
print("\nAfter:")
print(json.dumps(after, ensure_ascii=False, indent=2))


In [ ]:
RUN_CACHE_CLEAR_DEMO = False

if RUN_CACHE_CLEAR_DEMO:
    async with Client(mcp) as client:
        cleared = await call_json(client, "clear_metadata_cache")
    print(json.dumps(cleared, ensure_ascii=False, indent=2))
else:
    print("Skipping clear_metadata_cache demo. Set RUN_CACHE_CLEAR_DEMO = True to run it.")


## Paged Valid References

`get_valid_references_json` avoids returning a huge raw XML document when you only need a slice of citation URNs. `count_valid_references` gives a quick size check for a work or edition.


In [ ]:
ILIAD_CTS_EDITION = "urn:cts:greekLit:tlg0012.tlg001.perseus-grc1"

async with Client(mcp) as client:
    ref_count = await call_json(
        client,
        "count_valid_references",
        {"urn": ILIAD_CTS_EDITION, "level": 1},
    )
    ref_page = await call_json(
        client,
        "get_valid_references_json",
        {"urn": ILIAD_CTS_EDITION, "level": 1, "limit": 5, "offset": 0},
    )

assert ref_count["total_count"] >= ref_page["returned_count"]
assert ref_page["returned_count"] <= 5

print(json.dumps(ref_count, ensure_ascii=False, indent=2))
print(json.dumps(ref_page, ensure_ascii=False, indent=2))


## Paginated and Server-Scoped Library Search

`search_perseus` now supports `page_num`, `text_group`, `work`, and `result_format`. These are sent to Scaife's library search endpoint, so they filter before results come back to the MCP server.


In [ ]:
async with Client(mcp) as client:
    scoped_search = await call_json(
        client,
        "search_perseus",
        {
            "query": "μῆνιν",
            "language": "greek",
            "query_format": "unicode",
            "search_kind": "form",
            "page_num": 1,
            "text_group": "urn:cts:greekLit:tlg0012",
            "work": "urn:cts:greekLit:tlg0012.tlg001",
            "result_format": "instances",
        },
    )

summary = summarize_search(scoped_search)
assert scoped_search["total_count"] >= len(scoped_search.get("results", []))
assert summary["first_urn"] is None or "tlg0012.tlg001" in summary["first_urn"]

print(json.dumps(summary, ensure_ascii=False, indent=2))


## Author-Scoped Search

When `author` resolves to a single CTS textgroup, the server sends that textgroup to Scaife as a server-side filter. The response includes `author_scope` metadata explaining how the scope was applied.


In [ ]:
async with Client(mcp) as client:
    homer_search = await call_json(
        client,
        "search_perseus",
        {
            "query": '"μῆνιν ἄειδε"',
            "language": "greek",
            "query_format": "unicode",
            "preserve_operators": True,
            "author": "Homer",
        },
    )

assert "author_scope" in homer_search
print(json.dumps(summarize_search(homer_search), ensure_ascii=False, indent=2))


## Lemma and Operator-Preserving Search

`search_kind="lemma"` asks Scaife to search lemmas. `preserve_operators=True` keeps operator syntax such as quotes, `-`, `|`, `*`, and `~` intact instead of letting Beta Code auto-detection consume those characters.


In [ ]:
async with Client(mcp) as client:
    lemma_or_search = await call_json(
        client,
        "search_perseus",
        {
            "query": "λόγος | ἀνήρ",
            "language": "greek",
            "query_format": "unicode",
            "search_kind": "lemma",
            "preserve_operators": True,
        },
    )

assert lemma_or_search["total_count"] >= len(lemma_or_search.get("results", []))
print(json.dumps(summarize_search(lemma_or_search), ensure_ascii=False, indent=2))


## Search Within One Text

`search_within_text` uses Scaife's reader search endpoint and scopes the query to one text/edition URN. This is useful when you already know which edition you want to inspect.


In [ ]:
ILIAD_SCAIFE_EDITION = "urn:cts:greekLit:tlg0012.tlg001.perseus-grc2"

async with Client(mcp) as client:
    within_text = await call_json(
        client,
        "search_within_text",
        {
            "query": "μῆνιν",
            "text_urn": ILIAD_SCAIFE_EDITION,
            "language": "greek",
            "query_format": "unicode",
            "search_kind": "form",
            "size": 5,
            "offset": 0,
        },
    )

assert within_text["total_count"] >= len(within_text.get("results", []))
print(json.dumps(summarize_search(within_text), ensure_ascii=False, indent=2))


## Passage Highlights

`get_passage_highlights` asks Scaife for token highlight positions for one passage. The response includes `highlights`, where each entry contains the matched token text and token index.


In [ ]:
ILIAD_FIRST_LINE_SCAIFE = "urn:cts:greekLit:tlg0012.tlg001.perseus-grc2:1.1"

async with Client(mcp) as client:
    highlights = await call_json(
        client,
        "get_passage_highlights",
        {
            "query": "μῆνιν",
            "passage_urn": ILIAD_FIRST_LINE_SCAIFE,
            "language": "greek",
            "query_format": "unicode",
        },
    )

assert highlights["total_count"] >= 1
assert "highlights" in highlights["results"][0]
print(json.dumps(highlights, ensure_ascii=False, indent=2)[:1500])


## Scaife Metadata and Text Tools

The Scaife-specific tools are useful when Scaife search returns an edition URN that may not be available through Perseus CTS under the same identifier. They let you retrieve Scaife's own metadata, passage JSON, and passage plaintext directly.


In [ ]:
async with Client(mcp) as client:
    library_metadata = await call_json(
        client,
        "get_scaife_library_metadata",
        {"urn": ILIAD_SCAIFE_EDITION},
    )
    passage_json = await call_json(
        client,
        "get_scaife_passage_json",
        {"urn": ILIAD_FIRST_LINE_SCAIFE},
    )
    passage_text = await call_text(
        client,
        "get_scaife_passage_text",
        {"urn": ILIAD_FIRST_LINE_SCAIFE},
    )

assert library_metadata["urn"] == ILIAD_SCAIFE_EDITION
assert passage_json["urn"] == ILIAD_FIRST_LINE_SCAIFE
assert "μῆνιν" in passage_text

print("Library label:", library_metadata.get("label"))
print("Passage URN:", passage_json.get("urn"))
print("Passage text:", passage_text.strip())


## Summary

If all assertion cells pass, the notebook has exercised the expanded search and cache surface through the same MCP interface used by external clients. Because these cells call live upstream services, counts and first-result ordering may change as Scaife/Perseus data changes.
